# Berry–Esseen: b4-q2 — CPU, logi i checkpointy

Wybierz **Środowisko wykonawcze → Uruchom wszystko**. Zadanie jest już ustawione.
Google może poprosić o zgodę na Dysk, aby zachować checkpointy po utracie sesji.
Nie zmieniaj kodu i nie uruchamiaj dwóch kopii tego samego zadania równocześnie.

Najpierw krótki pomiar, potem do 3 godzin przydzielonego wyszukiwania w odcinkach.
Każdy odcinek ma zewnętrzny limit czasu. Kompletne drzewo przechodzi osobny akceptor.
Postęp jest widoczny w komórce oraz w plikach. Wynik częściowy zachowuje obie kopie checkpointu.
Ten notebook dotyczy jednego warunkowego drzewa n25, nie całego twierdzenia.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, time
CODE_COMMIT = "3a9935abd7337f5e0ce5b5e798a9c88a064eb786"
TASK = "b4-q2"
REPOSITORY = "https://github.com/rodziewiczk/berry-esseen-end2end.git"
CODE = Path("/content") / ("berry-esseen-code-" + CODE_COMMIT[:12])
print("Pobieram przypięty kod:", CODE_COMMIT, flush=True)
if not CODE.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(CODE)], check=True)
    subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", "--detach", CODE_COMMIT], check=True)
observed = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if observed != CODE_COMMIT:
    raise RuntimeError("Niezgodny commit źródeł")
subprocess.run(["git", "-C", str(CODE), "diff", "--exit-code", "HEAD", "--"], check=True)
sys.path.insert(0, str(CODE))
from be_replay.environment import snapshot
print(json.dumps(snapshot(), indent=2), flush=True)
print("Kod i środowisko odczytane. To jeszcze nie akceptacja matematyczna.", flush=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DURABLE = Path("/content/drive/MyDrive/Berry-Esseen-end2end")
DURABLE.mkdir(parents=True, exist_ok=True)
print("Logi i checkpointy:", DURABLE, flush=True)


In [ ]:
from datetime import datetime, timezone
SELFTEST = DURABLE / ("selftest-" + TASK + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".log")
print("Sprawdzam nadzorcę na rzeczywistym Linuksie. To lekkie testy, bez dowodu.", flush=True)
with SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                              "-s", "tests", "-p", "test_linux_supervisor.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Test nadzorcy nie przeszedł; obliczenia nie wystartują. Log: " + str(SELFTEST))
print("Test nadzorcy zakończony. Rozpoczynam właściwe zadanie.", flush=True)


In [ ]:
from be_replay.colab_batch import run_count
RESULT = run_count("b4-q2", durable_root=DURABLE,
                   total_seconds=10800, slice_seconds=1800, backup_interval=60)


In [ ]:
from IPython.display import display, FileLink
from google.colab import files
print("Stan:", RESULT["status"]["state"], flush=True)
print("Globalny dowód nie jest oznaczony jako zamknięty.", flush=True)
print("ZIP na Dysku:", RESULT["bundle"], flush=True)
display(FileLink(RESULT["bundle"]))
try:
    files.download(RESULT["bundle"])
except Exception as error:
    print("Pobierz ZIP z podanej ścieżki na Dysku:", error, flush=True)
